# 🎯 한국어 대화 요약 - SimPO (Simple Preference Optimization)

## 이 노트북은?
- **SFT 학습 후, 선호도 학습(SimPO)을 추가**하여 요약 품질을 개선
- SimPO는 "좋은 요약 vs 나쁜 요약"을 비교하여 모델이 더 좋은 요약을 생성하도록 학습
- **3단계 파이프라인**: SFT → Rejected 생성 → SimPO 학습

### 성능 비교
| 방법 | Mecab ROUGE-1 | 비고 |
|------|--------------|------|
| Baseline SFT | ~0.32 | 전체 시퀀스 학습 |
| Response-Only SFT | 0.5641 | 프롬프트 제외, 요약만 학습 |
| **SFT + SimPO (이 노트북)** | ~0.33 | 선호도 학습 추가 |
| Response-Only + MBR 앙상블 | 0.5716 | 최고 성능 |

### SimPO란?
- **S**imple **P**reference **O**ptimization
- DPO(Direct Preference Optimization)의 변형
- 별도의 Reference Model 없이 선호도 학습 가능 → 메모리 절약
- "좋은 응답(chosen)"과 "나쁜 응답(rejected)"의 쌍으로 학습

### 파이프라인 구조
```
Step 1: SFT 학습 (기본 요약 능력 학습)
    ↓
Step 2: SFT 모델로 train set에 대한 요약 생성 (= rejected)
         골드 요약 = chosen
    ↓
Step 3: SimPO 학습 (chosen vs rejected 비교 학습)
    ↓
Step 4: 추론 & 제출
```

---

## 0. 환경 설정

In [ ]:
# 이미 설치되어 있다면 건너뛰세요
# !pip install unsloth peft transformers datasets trl rouge pandas mecab-python3

In [ ]:
import os
import re
import json
import random
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from tqdm import tqdm

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# ===== 하이퍼파라미터 =====
MODEL_NAME = "unsloth/Qwen3-14B"
MAX_SEQ_LENGTH = 2048

# SFT 학습 설정
SFT_LORA_R = 16
SFT_LORA_ALPHA = 16
SFT_LR = 2e-4
SFT_EPOCHS = 3
SFT_BATCH = 1          # RTX 3090
SFT_GRAD_ACCUM = 32    # 실효 배치 = 32
SFT_OUTPUT = "outputs/simpo_sft"

# SimPO 학습 설정
SIMPO_LORA_R = 64
SIMPO_LORA_ALPHA = 64
SIMPO_LR = 5e-7        # SimPO는 매우 낮은 학습률 사용
SIMPO_EPOCHS = 1
SIMPO_BATCH = 1         # RTX 3090
SIMPO_GRAD_ACCUM = 4
SIMPO_GAMMA = 0.5       # SimPO 마진 파라미터
SIMPO_OUTPUT = "outputs/simpo_final"

# 추론 설정
MAX_NEW_TOKENS = 192
DATA_DIR = "data"

# 프롬프트
SYSTEM_PROMPT = (
    "당신은 한국어 대화 요약 전문가입니다. "
    "대화에는 #Person1#, #Person2# 등의 화자 태그가 사용됩니다. "
    "요약할 때 이 화자 태그를 그대로 사용하여 누가 무엇을 했는지 명확히 구분해주세요. "
    "핵심 내용만 1~3문장으로 간결하게 요약하세요."
)
USER_TEMPLATE = (
    "아래 대화를 읽고 핵심 내용을 요약해주세요. "
    "화자 태그(#Person1# 등)를 유지하세요.\n\n{dialogue}"
)

## Step 1: SFT 학습 (기본 요약 능력)

먼저 기본적인 SFT 학습을 합니다. 이미 SFT 체크포인트가 있다면 이 단계를 건너뛰세요.

> **참고**: 제공된 체크포인트(`baseline_ckpt/`)를 사용하면 이 단계를 건너뛸 수 있습니다.

In [ ]:
# SFT 체크포인트가 이미 있으면 건너뛰기
SFT_CHECKPOINT = f"{SFT_OUTPUT}/lora_adapter"

# 제공된 체크포인트 사용 (권장 - SFT 학습 시간 절약)
# SFT_CHECKPOINT = "baseline_ckpt"  # 제공된 SFT 체크포인트

SKIP_SFT = os.path.exists(SFT_CHECKPOINT)
print(f"SFT 체크포인트: {SFT_CHECKPOINT}")
print(f"SFT 학습 {'건너뛰기 (이미 존재)' if SKIP_SFT else '필요'}")

In [ ]:
if not SKIP_SFT:
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from trl import SFTTrainer, SFTConfig

    # 모델 로드
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=SFT_LORA_R, lora_alpha=SFT_LORA_ALPHA, lora_dropout=0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=42,
    )

    # 데이터
    train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
    def format_sft(row):
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(dialogue=row["dialogue"])},
            {"role": "assistant", "content": str(row["summary"])},
        ]
        return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=False)}

    train_dataset = Dataset.from_pandas(train_df).map(format_sft)

    # 학습
    trainer = SFTTrainer(
        model=model, train_dataset=train_dataset, processing_class=tokenizer,
        args=SFTConfig(
            output_dir=SFT_OUTPUT,
            per_device_train_batch_size=SFT_BATCH, gradient_accumulation_steps=SFT_GRAD_ACCUM,
            num_train_epochs=SFT_EPOCHS, learning_rate=SFT_LR,
            lr_scheduler_type="cosine", warmup_ratio=0.05, weight_decay=0.01,
            fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
            logging_steps=10, save_strategy="epoch", save_total_limit=2,
            optim="adamw_8bit", max_seq_length=MAX_SEQ_LENGTH, seed=42, report_to="none",
        ),
    )
    trainer.train()
    model.save_pretrained(SFT_CHECKPOINT)
    tokenizer.save_pretrained(SFT_CHECKPOINT)
    print(f"SFT 모델 저장: {SFT_CHECKPOINT}")

    # 메모리 해제
    del model, tokenizer, trainer
    import gc; gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"SFT 체크포인트 사용: {SFT_CHECKPOINT}")

## Step 2: Rejected 데이터 생성

SimPO 학습에는 **chosen(좋은 요약)** vs **rejected(나쁜 요약)** 쌍이 필요합니다.

- **chosen**: 골드 요약 (data/train.csv의 summary)
- **rejected**: SFT 모델이 생성한 요약 (학습 전 모델의 출력이므로 품질이 낮음)

SFT 모델로 train 데이터의 대화를 요약하면, 골드보다 품질이 낮은 요약이 생성됩니다.
이 쌍을 사용하여 "골드처럼 좋은 요약을 생성하도록" 학습합니다.

In [ ]:
REJECT_DATA_PATH = f"{DATA_DIR}/train_with_rejects.json"

SKIP_REJECT = os.path.exists(REJECT_DATA_PATH)
print(f"Rejected 데이터: {REJECT_DATA_PATH}")
print(f"생성 {'건너뛰기 (이미 존재)' if SKIP_REJECT else '필요'}")

In [ ]:
def postprocess(text):
    """모델 출력 후처리"""
    text = re.sub(r"<\|.*?\|>", "", text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"#\s*Person\s*(\d+)\s*#", r"#Person\1#", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"^요약\s*:\s*", "", text).strip()
    return text

In [ ]:
if not SKIP_REJECT:
    from unsloth import FastLanguageModel
    from peft import PeftModel

    # SFT 모델 로드
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True,
    )
    model = PeftModel.from_pretrained(model, SFT_CHECKPOINT, is_trainable=False)
    FastLanguageModel.for_inference(model)
    print(f"SFT 모델 로드: {SFT_CHECKPOINT}")

    # Train 데이터에 대한 요약 생성
    train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
    results = []

    for i in tqdm(range(len(train_df)), desc="Rejected 생성"):
        row = train_df.iloc[i]
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(dialogue=row["dialogue"])},
        ]
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                           max_length=MAX_SEQ_LENGTH).to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        generated = outputs[0][inputs["input_ids"].shape[1]:]
        rejected = tokenizer.decode(generated, skip_special_tokens=True).strip()
        rejected = postprocess(rejected)

        results.append({
            "dialogue": row["dialogue"],
            "chosen": str(row["summary"]),   # 골드 요약 = chosen
            "rejected": rejected,             # 모델 생성 = rejected
        })

    # 저장
    with open(REJECT_DATA_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"저장: {REJECT_DATA_PATH} ({len(results)}건)")

    # 예시
    for i in range(3):
        print(f"\n--- 예시 {i+1} ---")
        print(f"  Chosen:   {results[i]['chosen'][:100]}")
        print(f"  Rejected: {results[i]['rejected'][:100]}")

    # 메모리 해제
    del model, tokenizer
    import gc; gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"Rejected 데이터 사용: {REJECT_DATA_PATH}")
    with open(REJECT_DATA_PATH, "r") as f:
        results = json.load(f)
    print(f"데이터: {len(results)}건")
    print(f"\n예시:")
    print(f"  Chosen:   {results[0]['chosen'][:100]}")
    print(f"  Rejected: {results[0]['rejected'][:100]}")

## Step 3: SimPO 학습

### SimPO의 핵심 아이디어

DPO와 달리 SimPO는 **Reference Model이 필요 없습니다**:

$$\mathcal{L}_{\text{SimPO}} = -\log\sigma\left(\frac{\beta}{|y_w|}\log p(y_w|x) - \frac{\beta}{|y_l|}\log p(y_l|x) - \gamma\right)$$

- $y_w$: chosen (좋은 요약), $y_l$: rejected (나쁜 요약)
- $\gamma$: 마진 파라미터 (chosen과 rejected 사이의 최소 차이)
- 길이로 정규화 ($|y|$)하여 짧은 응답에 bias 방지

### 학습 과정
1. SFT LoRA를 베이스 모델에 **merge** (합침)
2. 새 LoRA 어댑터를 추가하여 SimPO 학습
3. 이렇게 하면 SFT 기반 위에 선호도 학습이 추가됨

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import AutoModelForCausalLM, AutoTokenizer

# Step 3-1: SFT LoRA를 base에 merge
# SimPO는 SFT 위에 새 LoRA를 추가하므로, 먼저 SFT를 base에 합쳐야 합니다.

MERGED_PATH = f"{SIMPO_OUTPUT}/_merged_sft"

if not os.path.exists(os.path.join(MERGED_PATH, "config.json")):
    print("SFT LoRA를 base 모델에 merge 중...")
    os.makedirs(MERGED_PATH, exist_ok=True)

    # fp16으로 로드 (4bit에서는 merge 불안정)
    _model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
    )
    _tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    from peft import PeftModel as _PeftModel
    _model = _PeftModel.from_pretrained(_model, SFT_CHECKPOINT)
    print("LoRA merge 중...")
    _model = _model.merge_and_unload()

    print(f"저장 중: {MERGED_PATH}")
    _model.save_pretrained(MERGED_PATH)
    _tokenizer.save_pretrained(MERGED_PATH)
    print("Merge 완료!")

    del _model, _tokenizer
    import gc; gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"Merged 모델 이미 존재: {MERGED_PATH}")

In [ ]:
# Step 3-2: Merged 모델을 4bit로 로드 + 새 LoRA for SimPO
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MERGED_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
print(f"Merged SFT 모델 로드 완료")

model = FastLanguageModel.get_peft_model(
    model,
    r=SIMPO_LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=SIMPO_LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"SimPO 학습 파라미터: {trainable:,}")

In [ ]:
# Step 3-3: 선호도 데이터 준비
with open(REJECT_DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Preference 데이터: {len(raw_data)}건")

def format_for_simpo(item):
    """SimPO 형식으로 변환: chosen/rejected 메시지 리스트"""
    prompt_msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(dialogue=item["dialogue"])},
    ]
    chosen_msgs = prompt_msgs + [{"role": "assistant", "content": item["chosen"]}]
    rejected_msgs = prompt_msgs + [{"role": "assistant", "content": item["rejected"]}]
    return {"chosen": chosen_msgs, "rejected": rejected_msgs}

formatted = [format_for_simpo(item) for item in raw_data]

# 품질 필터링
filtered = []
for item in formatted:
    chosen_text = item["chosen"][-1]["content"].strip()
    rejected_text = item["rejected"][-1]["content"].strip()
    # chosen과 동일하면 학습 의미 없음
    if chosen_text == rejected_text:
        continue
    # rejected가 너무 짧으면 제외
    if len(rejected_text) < 5:
        continue
    # rejected가 chosen의 3배 이상이면 대화 복사로 판단
    if len(rejected_text) > max(len(chosen_text) * 3, 300):
        continue
    filtered.append(item)

print(f"필터링 후: {len(filtered)}건 (제거: {len(formatted) - len(filtered)}건)")
dataset = Dataset.from_list(filtered)

In [ ]:
# Step 3-4: SimPO 학습 실행
from trl import CPOConfig, CPOTrainer

os.makedirs(SIMPO_OUTPUT, exist_ok=True)

simpo_trainer = CPOTrainer(
    model=model,
    args=CPOConfig(
        per_device_train_batch_size=SIMPO_BATCH,
        gradient_accumulation_steps=SIMPO_GRAD_ACCUM,
        num_train_epochs=SIMPO_EPOCHS,
        learning_rate=SIMPO_LR,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        output_dir=SIMPO_OUTPUT,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        seed=42,
        report_to="none",
        # SimPO 전용 설정
        loss_type="simpo",       # SimPO 손실 함수 사용
        cpo_alpha=0.0,            # CPO alpha (SimPO에서는 0)
        simpo_gamma=SIMPO_GAMMA,  # 마진 파라미터
        max_length=MAX_SEQ_LENGTH,
        max_prompt_length=1800,
        save_strategy="epoch",
        save_total_limit=2,
    ),
    train_dataset=dataset,
    processing_class=tokenizer,
)

print(f"SimPO 학습 시작")
print(f"  데이터: {len(filtered)}건")
print(f"  LR: {SIMPO_LR}, Gamma: {SIMPO_GAMMA}")
print(f"  배치: {SIMPO_BATCH} x {SIMPO_GRAD_ACCUM} = {SIMPO_BATCH * SIMPO_GRAD_ACCUM}")

simpo_trainer.train()

# 저장
simpo_adapter_path = f"{SIMPO_OUTPUT}/lora_adapter"
model.save_pretrained(simpo_adapter_path)
tokenizer.save_pretrained(simpo_adapter_path)
print(f"SimPO 모델 저장: {simpo_adapter_path}")

## Step 4: 추론 & 평가

SimPO 모델은 2단계 로드가 필요합니다:
1. Merged SFT 모델 로드 (base + SFT LoRA가 합쳐진 모델)
2. SimPO LoRA 어댑터 로드

In [ ]:
# 추론을 위한 모델 로드
# (학습 직후라면 위의 model을 그대로 사용 가능)
# 새로 로드하려면 아래 코드 사용:

# from unsloth import FastLanguageModel
# from peft import PeftModel
#
# # Merged SFT 모델 로드
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=MERGED_PATH,  # SFT가 merge된 모델
#     max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True,
# )
# # SimPO LoRA 로드
# model = PeftModel.from_pretrained(model, simpo_adapter_path, is_trainable=False)

FastLanguageModel.for_inference(model)
print("추론 모드 전환 완료")

In [ ]:
def generate_summary(dialogue):
    """대화 하나에 대한 요약 생성"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(dialogue=dialogue)},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    summary = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return postprocess(summary)

In [ ]:
# Dev 평가
dev_df = pd.read_csv(f"{DATA_DIR}/dev.csv")
print(f"Dev 추론 ({len(dev_df)}개)")

dev_preds = []
for i in tqdm(range(len(dev_df))):
    pred = generate_summary(dev_df.iloc[i]["dialogue"])
    dev_preds.append(pred if pred.strip() else "빈 요약")

# ROUGE 계산
from rouge import Rouge
import mecab

m = mecab.MeCab()
rouge = Rouge()

golds = [str(s).strip() if str(s).strip() else "빈 요약" for s in dev_df["summary"]]

# Mecab ROUGE (대회 공식)
preds_m = [" ".join(m.morphs(p)) for p in dev_preds]
golds_m = [" ".join(m.morphs(g)) for g in golds]
mc_scores = rouge.get_scores(preds_m, golds_m, avg=True)

print(f"\n{'='*60}")
print(f"[SimPO] Dev Mecab ROUGE:")
print(f"  ROUGE-1: {mc_scores['rouge-1']['f']:.4f}")
print(f"  ROUGE-2: {mc_scores['rouge-2']['f']:.4f}")
print(f"  ROUGE-L: {mc_scores['rouge-l']['f']:.4f}")
print(f"{'='*60}")

In [ ]:
# Test 추론 & 제출 파일 생성
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print(f"Test 추론 ({len(test_df)}개)")

test_preds = []
for i in tqdm(range(len(test_df))):
    pred = generate_summary(test_df.iloc[i]["dialogue"])
    test_preds.append(pred if pred.strip() else "빈 요약")

# 제출 파일
os.makedirs("prediction", exist_ok=True)
submission = pd.DataFrame({"fname": test_df["fname"], "summary": test_preds})
submission.to_csv("prediction/submission_simpo.csv", index=False)
print(f"\n제출 파일: prediction/submission_simpo.csv ({len(submission)}행)")
submission.head()

## 제공된 체크포인트로 바로 추론하기

학습 없이 제공된 SimPO 체크포인트로 바로 추론하려면 아래 코드를 사용하세요.

**주의**: SimPO 모델은 SFT가 merge된 모델 위에 LoRA를 추가한 구조입니다.
제공된 체크포인트를 사용하려면 먼저 SFT merge 과정(Step 3-1)이 필요합니다.

In [ ]:
# ===== 제공된 체크포인트로 바로 추론 =====
# 아래 코드는 학습 없이 제공된 체크포인트만으로 추론할 때 사용하세요.

# from unsloth import FastLanguageModel
# from peft import PeftModel
#
# # Step 1: SFT를 merge한 모델이 필요 (Step 3-1의 MERGED_PATH)
# # 이미 생성했다면 아래 경로 사용
# MERGED_PATH = "outputs/simpo_final/_merged_sft"
#
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=MERGED_PATH,
#     max_seq_length=2048,
#     load_in_4bit=True,
# )
#
# # Step 2: SimPO LoRA 로드
# model = PeftModel.from_pretrained(model, "simpo_v2_ckpt", is_trainable=False)
# FastLanguageModel.for_inference(model)
#
# # 이후 generate_summary() 함수로 추론 가능

## 📊 분석 & 정리

### SimPO의 장단점

**장점:**
- Reference Model 불필요 → DPO 대비 메모리 절약
- 길이 정규화로 짧은 응답 bias 방지
- SFT 기반 위에 선호도 학습 추가 가능

**단점:**
- SFT → Reject 생성 → SimPO 3단계 파이프라인 필요
- SFT 품질이 낮으면 rejected 품질도 낮아 학습 효과 제한
- 이 대회에서는 Response-Only Loss가 더 큰 효과

### 성능 비교 정리
| 방법 | Mecab ROUGE-1 | 핵심 기법 |
|------|--------------|----------|
| Baseline SFT (r=16) | ~0.32 | 전체 시퀀스 학습 |
| SFT + SimPO | ~0.33 | 선호도 학습 추가 |
| **Response-Only SFT (r=32)** | **0.5641** | **프롬프트 마스킹** |
| RO-SFT + MBR 앙상블 | 0.5716 | 8개 프롬프트 합의 |

### 핵심 인사이트
이 대회에서 가장 큰 성능 향상은 **Response-Only Loss**에서 왔습니다 (0.32 → 0.56, +75%).
SimPO는 소폭 개선(~0.01)에 그쳤는데, 이는 SFT가 이미 좋은 요약을 생성하고 있어
rejected와 chosen의 차이가 크지 않았기 때문입니다.

**추천 학습 순서:**
1. `대화요약_Baseline_SFT.ipynb` → 기본 이해
2. `대화요약_Qwen3_14B_LoRA_SFT.ipynb` → Response-Only Loss (핵심!)
3. `대화요약_SimPO.ipynb` (이 노트북) → 선호도 학습 이해